# 23 — Knowledge Management
> Ask it a question about past work. It searches your engagement documents, pulls the most relevant precedents, and drafts a grounded brief that cites them explicitly — no hallucinated references.

*Run all cells. Swap the input in the final code cell with your own data.*

In [ ]:
%pip install -q langchain-openai langchain-core pydantic python-dotenv
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI


# --- Schema ---

class Precedent(BaseModel):
    title: str = Field(description="Document title or engagement name.")
    excerpt: str = Field(description="Relevant extract or summary from the document.")
    relevance_reason: str = Field(
        description="Why this precedent is relevant to the current query."
    )
    source_id: str = Field(description="Identifier linking back to the source document.")


class KnowledgeBrief(BaseModel):
    query: str = Field(description="The original user query or drafting task.")
    precedents: list[Precedent] = Field(
        description="Retrieved precedents most relevant to the query, ranked by relevance."
    )
    synthesis: str = Field(
        description=(
            "A structured synthesis that draws on the precedents to address the query. "
            "Must explicitly cite precedents by title when drawing on them."
        )
    )
    gaps: list[str] = Field(
        description="Any gaps where no precedent covers the query and original work is needed."
    )


# --- Corpus (swap this out for your own documents) ---

CORPUS: list[dict[str, str]] = [
    {
        "id": "DOC-001",
        "title": "EMEA SaaS Go-to-Market Playbook (2022)",
        "summary": (
            "Covers channel strategy, pricing localisation, and regulatory considerations "
            "for B2B SaaS expansion across the UK, Germany, and France. Key finding: "
            "partner-led motion outperformed direct sales in mid-market segment by 2.4x."
        ),
    },
    {
        "id": "DOC-002",
        "title": "FinTech Regulatory Compliance Engagement (2023)",
        "summary": (
            "Assessment of PSD2 and GDPR obligations for a payment processing client. "
            "Produced a gap analysis, remediation roadmap, and data flow mapping. "
            "Timeline: 14 weeks from scoping to final report."
        ),
    },
    {
        "id": "DOC-003",
        "title": "Manufacturing Digital Transformation -- Phase 1 Close-out (2023)",
        "summary": (
            "IoT sensor rollout across three production facilities. Lessons learned: "
            "change management was the critical path, not technology. Recommend a "
            "dedicated change champion per site for any future phase."
        ),
    },
    {
        "id": "DOC-004",
        "title": "Investor Readiness Review -- Series B Candidate (2024)",
        "summary": (
            "Assessed a SaaS HR-tech company against typical Series B investor criteria. "
            "Key gaps: weak unit economics narrative, no clear path to profitability in "
            "18 months. Recommendations: restructure MRR/CAC/LTV story, prepare a "
            "financial bridge scenario."
        ),
    },
    {
        "id": "DOC-005",
        "title": "Strategic Sourcing Review -- Global Logistics Provider (2022)",
        "summary": (
            "Benchmark of procurement function across 12 categories. Identified "
            "$14M in savings opportunities. Recommended a category management operating "
            "model and supplier consolidation in packaging and freight."
        ),
    },
]

_INDEX = "\n".join(
    f'[{doc["id"]}] {doc["title"]}: {doc["summary"]}' for doc in CORPUS
)

_RETRIEVAL_SYSTEM = SystemMessage(
    f"""You are a knowledge retrieval specialist. Given a query, select the 2-4 most
relevant documents from the following corpus index.

For each selected document, return:
- source_id: the document ID (e.g. DOC-001)
- title: as listed
- excerpt: the most relevant sentence or phrase from the summary
- relevance_reason: one sentence explaining why this document is relevant to the query

Only select documents that genuinely address the query.

Corpus:
{_INDEX}

Return a KnowledgeBrief with an empty synthesis and empty gaps for now.
Set query to the user's query."""
)

_SYNTHESIS_SYSTEM = SystemMessage(
    """You are a senior knowledge management analyst. Given a query and a set of
retrieved precedents, synthesise a concise, structured response that:
1. Directly addresses the query.
2. Explicitly cites precedents by title when drawing on them.
3. Notes any gaps where the precedents do not cover the query.

Return a KnowledgeBrief object."""
)


def _retrieve(query: str) -> list[Precedent]:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    result: KnowledgeBrief = llm.with_structured_output(KnowledgeBrief).invoke(
        [
            _RETRIEVAL_SYSTEM,
            {"role": "user", "content": f"Query: {query}"},
        ]
    )
    return result.precedents


def run(query: str) -> KnowledgeBrief:
    """Retrieve relevant precedents and synthesise a grounded knowledge brief."""
    precedents = _retrieve(query)

    precedent_text = "\n\n".join(
        f"[{p.source_id}] {p.title}\n{p.excerpt}\nRelevance: {p.relevance_reason}"
        for p in precedents
    ) or "No precedents retrieved."

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    brief: KnowledgeBrief = llm.with_structured_output(KnowledgeBrief).invoke(
        [
            _SYNTHESIS_SYSTEM,
            {
                "role": "user",
                "content": (
                    f"Query: {query}\n\n"
                    f"Precedents retrieved:\n{precedent_text}"
                ),
            },
        ]
    )
    brief.precedents = precedents
    return brief


print("Ready.")

## The scenario

A fintech client is asking for help preparing for a Series B raise. Before the kick-off call, the partner wants to know what the firm has already done in adjacent areas — investor readiness, regulatory exposure, and SaaS financials — so the team walks in with informed questions rather than starting from scratch.

The agent scans the engagement library, surfaces the most relevant prior work, and drafts a ready-to-share brief that cites each source by name.

In [ ]:
query = (
    "A fintech client is preparing for a Series B raise and needs help sharpening "
    "their investor story. What does our prior work tell us about what investors "
    "look for at this stage, and are there any regulatory angles we should raise early?"
)

brief = run(query)

print(f"QUERY\n{'─' * 60}")
print(brief.query)

print(f"\nPRECEDENTS RETRIEVED ({len(brief.precedents)})\n{'─' * 60}")
for p in brief.precedents:
    print(f"  [{p.source_id}]  {p.title}")
    print(f"  Why it matters: {p.relevance_reason}")
    print()

print(f"SYNTHESIS\n{'─' * 60}")
print(brief.synthesis)

if brief.gaps:
    print(f"\nGAPS — ORIGINAL WORK NEEDED ({len(brief.gaps)})\n{'─' * 60}")
    for g in brief.gaps:
        print(f"  - {g}")

## Use your own data

Replace the input above with:
- **Your query** — any question about past work, phrased in plain language
- **Your corpus** — swap the `CORPUS` list in Cell 2 with your own engagement summaries (title, id, summary per document)

The agent will return the most relevant prior engagements, a cited synthesis, and a list of gaps where no prior work covers the question.